# Financial Impact Predictor — Training Notebook

This notebook trains a model to predict **actual stock price movement** (Up/Down) based on financial news articles, using the `financial-news-dataset`.

**Instructions:**
1. Upload this notebook to Google Colab.
2. Upload your `.json.xz` dataset files (e.g., `2023_processed.json.xz`) to Colab.
3. Go to **Runtime > Change runtime type** and select **T4 GPU**.
4. Run all cells.
5. Download the `saved_model` folder and use it in your Node/Flask app.

In [ ]:
!pip install transformers datasets accelerate torch lzma

### 1. Load and Preprocess Data
We calculate the real stock impact: `1` if the price went up the next day, `0` if it went down.

In [ ]:
import lzma
import json
import pandas as pd
import glob

# Look for the uploaded dataset files (update the path if you put them in a folder)
files = glob.glob('*.json.xz')
if not files:
    print("⚠️ Please upload the .json.xz files to Colab first!")

data = []
for file in files:
    print(f"Processing {file}...")
    with lzma.open(file, 'rt', encoding='utf-8') as f:
        articles = json.load(f)
        for article in articles:
            try:
                # Make sure we have text and mentioned companies
                if not article.get('maintext') or not article.get('mentioned_companies'):
                    continue
                
                ticker = article['mentioned_companies'][0]
                
                curr_price = article.get(f'curr_day_price_{ticker}')
                next_price = article.get(f'next_day_price_{ticker}')
                
                if curr_price is not None and next_price is not None and curr_price > 0:
                    # 1 if price went UP or stayed flat, 0 if price went DOWN
                    label = 1 if next_price >= curr_price else 0
                    
                    data.append({
                        'text': article['maintext'][:1000],  # Take first 1000 chars to avoid memory limits
                        'label': label
                    })
            except Exception as e:
                pass

df = pd.DataFrame(data)
print(f"\nExtracted {len(df)} articles with valid price data.")
print(df['label'].value_counts())

### 2. Balance the Dataset
If the market was mostly going up in 2023, the model will just guess "Up" every time. We must balance the classes.

In [ ]:
min_class_size = df['label'].value_counts().min()

# Sample equally from both classes
df_balanced = pd.concat([
    df[df['label'] == 1].sample(min_class_size, random_state=42),
    df[df['label'] == 0].sample(min_class_size, random_state=42)
])

# Shuffle the dataset
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Balanced dataset size: {len(df_balanced)}")

### 3. Prepare for HuggingFace Trainer

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Convert to HuggingFace Dataset format
hf_dataset = Dataset.from_pandas(df_balanced)

# Split into Train (80%) and Eval (20%)
split_dataset = hf_dataset.train_test_split(test_size=0.2)
train_data = split_dataset['train']
eval_data = split_dataset['test']

# Load FinBERT tokenizer
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=256)

tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_eval = eval_data.map(tokenize_function, batched=True)

### 4. Train the Model

In [ ]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score

# Load model with 2 labels (0=Down, 1=Up)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, ignore_mismatched_sizes=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

training_args = TrainingArguments(
    output_dir="./impact_predictor_checkpoints",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,       # Small learning rate prevents forgetting pre-trained knowledge
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,       # 3 epochs is plenty for fine-tuning
    weight_decay=0.01,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

# Start training!
trainer.train()

### 5. Save the Final Model

In [ ]:
import os
import shutil
from google.colab import files

output_dir = "./saved_model"
os.makedirs(output_dir, exist_ok=True)

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"\n✅ Model saved to {output_dir}/.")

# Zip the folder to make downloading easier
shutil.make_archive('my_finbert_model', 'zip', 'saved_model')
print("✅ Zipped model to my_finbert_model.zip. Initiating download...")

# Download the zip file
files.download('my_finbert_model.zip')